# Applying ML models for stock returns prediction

## Initialization

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, make_scorer


tickers = ["JNJ", "AAPL", "MSFT", "PG", "KO"]
start = "2017-01-01"
end = "2020-12-31"

stock_data = {}

In [8]:
# Download market indext data for S&P 500 index ticker
sp500 = yf.download("^GSPC", start=start, end=end, auto_adjust=True, progress=False)
sp500.columns = sp500.columns.droplevel(1)
sp500_market_return = sp500['Close'].pct_change().to_frame(name='market_return')

In [9]:
# Download stock data for each ticker and store it in a dictionary
for ticker in tickers:
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    # Drop stock name level from columns
    df.columns = df.columns.droplevel(1)
    # Ensure datetime index and sort by date
    df.index = pd.to_datetime(df.index)
    df.sort_index(inplace=True)
    # Calculate stock returns variable
    df['stock_return'] = df['Close'].pct_change()
    # Add market indext return to the stock data
    df = df.merge(sp500_market_return, left_index=True, right_index=True, how='inner')

    # Add the stock data to the dictionary
    stock_data[ticker] = df

df = stock_data['AAPL']
df

,Close,High,Low,Open,Volume,stock_return,market_return
Date,,,,,,,
2017-01-03,26.745852,26.787300,26.425776,26.665257,115127600,NaN,NaN
2017-01-04,26.715925,26.828759,26.653753,26.676780,84472400,-0.001119,0.005722
2017-01-05,26.851780,26.909347,26.667563,26.692893,88774400,0.005085,-0.000771
2017-01-06,27.151129,27.208696,26.819540,26.890923,127007600,0.011148,0.003517
2017-01-09,27.399818,27.501138,27.158036,27.160337,134247600,0.009159,-0.003549
...,...,...,...,...,...,...,...
2020-12-23,127.364151,128.793775,127.189086,128.531199,88223700,-0.006976,0.000746
2020-12-24,128.346420,129.795514,127.500313,127.714274,54930100,0.007712,0.003537
2020-12-28,132.936813,133.568960,129.844121,130.310952,124486200,0.035766,0.008723


## Preprocessing, Feature Engineering, Train/Test Spliting

### Preprocessing Functions

In [10]:
def handle_outliers_winsorize(df, col, lower_q=0.01, upper_q=0.99):
    lower = df[col].quantile(lower_q)
    upper = df[col].quantile(upper_q)
    df[col] = df[col].clip(lower, upper)
    return df

### Feature Engineering Functions

In [11]:
def add_moving_average(df, n, price_col='Close'):
    """
    Calculate n-day Simple Moving Average (SMA) on a financial time series.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing price data
    n : int
        Window size (e.g., 10, 50, 200)
    price_col : str
        Column name to calculate SMA on (default: 'Close')

    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with new SMA column added
    """
    
    df[f'SMA_{n}'] = df[price_col].rolling(window=n, min_periods=n).mean()
    return df

def add_volatility(df, n=20, price_col='Close', annualize=False):
    """
    Add an n-day rolling volatility feature based on past returns.

    This is a rolling-window feature engineering step:
    - returns describe day-to-day price changes
    - rolling std of returns estimates recent noise / uncertainty
    - if annualize=True, daily volatility is scaled by sqrt(252)

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing price data.
    n : int
        Rolling window size.
    price_col : str, default='Close'
        Column used to compute returns.
    annualize : bool, default=False
        If True, annualise daily volatility using sqrt(252).

    Returns
    -------
    pandas.DataFrame
        Copy of df with:
        - 'return'
        - f'volatility_{n}' (or annualised version)
    """

    # Rolling volatility = recent standard deviation of returns
    vol_col = f'volatility_{n}'
    df[vol_col] = df['stock_return'].rolling(window=n, min_periods=n).std()

    # Optional annualisation for finance applications
    if annualize:
        df[vol_col] = df[vol_col] * np.sqrt(252)

    return df

def add_momentum_rsi(df, n=14, price_col='Close', col_name=None):
    """
    Adds Relative Strength Index (RSI) feature to dataframe.

    Parameters:
    - df: pandas DataFrame
    - n: lookback window (default = 14, standard in finance)
    - price_col: column name for price (default = 'Close')
    - col_name: optional custom column name

    Returns:
    - df with RSI column added
    """
    
    if col_name is None:
        col_name = f'RSI_{n}'
    
    # Price changes (returns in price space, not percentage)
    delta = df[price_col].diff()

    # Gains and losses
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)

    gain = pd.Series(gain, index=df.index)
    loss = pd.Series(loss, index=df.index)

    # Rolling averages (Wilder's smoothing approximation using mean)
    avg_gain = gain.rolling(window=n, min_periods=n).mean()
    avg_loss = loss.rolling(window=n, min_periods=n).mean()

    # Avoid division by zero
    rs = avg_gain / (avg_loss + 1e-10)

    # RSI formula
    rsi = 100 - (100 / (1 + rs))

    df[col_name] = rsi

    return df

In [12]:
# Step 1 (Load Data): Load historical stock data

# Handle initiall missing values (calendar gaps) using forward fill
df['Close'].fillna(method='ffill', inplace=True) 
# Log-transform the Volume to reduce skewness
df['Volume'] = np.log1p(df['Volume'])

# Step 2 (Feature Engineering): Compute technical indicators
add_moving_average(df,10)
add_moving_average(df,50)
add_moving_average(df,200)
add_volatility(df)
add_momentum_rsi(df)

# Step 3: Define Features and Target Variable
X = df.drop(columns=['Close', 'Open', 'High', 'Low'])
y = df['stock_return'].shift(-1).to_frame(name='target')
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

# Step 4: Clean the Data
# 4.1. Handling Outliers
handle_outliers_winsorize(X,'stock_return')
handle_outliers_winsorize(X,'market_return')

# 4.2. Handling Type (Convert all columns to numeric (forcing errors to NaN))
X = X.apply(pd.to_numeric, errors='coerce')

# 4.3. Handling Infinities (Replace infinities with NaN)
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# 4.4. Re-indexing (Reset index before dropping NaNs to avoid index misalignment)
X.reset_index(inplace=True, drop=True)
y.reset_index(inplace=True, drop=True)

# 4.5. Handling Missing Values (Drop rows with NaN values in X and align y accordingly)
X.dropna(inplace=True)
y = y.loc[X.index]

# Drop NaNs in y (shift(-1) creates NaN at the last row; missing Close values can create more)
y.dropna(inplace=True)
X = X.loc[y.index]

# Step 5 Train-Test Split (Simple 75% Train, 25% Test)
split_index = int(0.75 * len(X))
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (604, 8)
y_train shape: (604,)
X_test shape: (202, 8)
y_test shape: (202,)


/var/folders/xq/dznpw7s50vdg77ddlcz18rvw0000gn/T/ipykernel_8563/1671362036.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Close'].fillna(method='ffill', inplace=True)
/var/folders/xq/dznpw7s50vdg77ddlcz18rvw0000gn/T/ipykernel_8563/1671362036.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Close'].fillna(method='ffill', inplace=True)


## Model

### SVM for Regression (SVR) 

In [34]:
def directional_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.sign(y_true) == np.sign(y_pred))


def build_svr_pipeline():
    return Pipeline(
        steps=[
            ("scaler", MinMaxScaler()),
            ("model", SVR()),
        ]
    )


def get_svr_param_grid():
    return {
        "model__kernel": ["linear", "rbf"],
        "model__C": [0.1, 1.0, 10.0, 100.0],
        "model__epsilon": [0.0001, 0.001, 0.01, 0.05],
        "model__gamma": ["scale", "auto", 0.01, 0.1, 1.0],
    }


def train_svr_with_timeseries_cv(
    X_train,
    y_train,
    n_splits=5,
    scoring_refit="mse",
    n_jobs=-1,
):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    pipeline = build_svr_pipeline()
    param_grid = get_svr_param_grid()

    scoring = {
        "mse": "neg_mean_squared_error",
        "r2": "r2",
        "da": make_scorer(directional_accuracy),
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit=scoring_refit,
        cv=tscv,
        n_jobs=n_jobs,
        verbose=0,
    )

    grid_search.fit(X_train, y_train) # Fit Scaler and train the model

    best_model = grid_search.best_estimator_
    best_idx = grid_search.best_index_

    best_model.fit(X_train, y_train)

    return {
        "model": best_model,
        "best_params": grid_search.best_params_,
        "MSE": -grid_search.cv_results_['mean_test_mse'][best_idx],
        "R2": grid_search.cv_results_['mean_test_r2'][best_idx],
        "DA": grid_search.cv_results_['mean_test_da'][best_idx],
        "grid_search": grid_search,
    }


def evaluate_regression_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "test_mse": mean_squared_error(y_test, y_pred),
        "test_r2": r2_score(y_test, y_pred),
        "test_da": directional_accuracy(y_test, y_pred),
    }

    return y_pred, metrics

svr_results = train_svr_with_timeseries_cv(
    X_train=X_train,
    y_train=y_train,
    n_splits=5,
)


print("Best Hyperparameters:")
print(svr_results["best_params"])

print("\nAverage CV Metrics:")
print({
    "MSE": svr_results["MSE"],
    "R2": svr_results["R2"],
    "DA": svr_results["DA"],
})  

svr_model = svr_results["model"]
svr_test_pred, svr_test_metrics = evaluate_regression_model(
    model=svr_model,
    X_test=X_test,
    y_test=y_test,
)

print("\nTest Set Metrics:")
print(svr_test_metrics)

Best Hyperparameters:
{'model__C': 0.1, 'model__epsilon': 0.01, 'model__gamma': 0.01, 'model__kernel': 'rbf'}

Average CV Metrics:
{'MSE': np.float64(0.00043249525571561174), 'R2': np.float64(-0.07141776995488373), 'DA': np.float64(0.47000000000000003)}

Test Set Metrics:
{'test_mse': 0.0007374385686642964, 'test_r2': 0.002195017930955112, 'test_da': np.float64(0.5742574257425742)}
